# Time Complexity and Execution Time Analysis

This notebook is the main analysis entrypoint for the execution-time experiment. It keeps the existing algorithm and utility modules, varies input size through stratified row subsampling, and keeps all dataset features available to the feature selection algorithms.

## 1. Imports and Experiment Constants

In [ ]:
from __future__ import annotations

import json
import time
from pathlib import Path
from typing import Callable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.sparse import issparse
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from algorithms.bga import binary_genetic_algorithm
from algorithms.bgwo import binary_grey_wolf_optimizer
from algorithms.bpso import binary_particle_swarm_optimization
from algorithms.bwoa import binary_whale_optimization_algorithm
from utils.data_loader import PreparedDataset, load_dataset
from utils.fitness import make_accuracy_function, make_fitness_function

DATASET_NAMES = ["madelon", "arcene", "dexter", "dorothea", "gisette"]
FRACTIONS = [0.2, 0.4, 0.6, 0.8, 1.0]
RANDOM_STATE = 42
POPULATION_SIZE = 10
ITERATIONS = 30
EARLY_STOPPING_PATIENCE = 5
MIN_DELTA = 0.001

RESULTS_DIR = Path("results") / "time_complexity_analysis"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

ALGORITHMS: dict[str, Callable[..., tuple[np.ndarray, float, list[float], bool]]] = {
    "BGA": binary_genetic_algorithm,
    "BPSO": binary_particle_swarm_optimization,
    "BGWO": binary_grey_wolf_optimizer,
    "BWOA": binary_whale_optimization_algorithm,
}

DATASET_CACHE: dict[str, tuple[np.ndarray, np.ndarray, np.ndarray]] = {}

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)


: 

## 2. Shared Helpers

In [ ]:
def dataset_seed(dataset_name: str) -> int:
    return RANDOM_STATE + DATASET_NAMES.index(dataset_name) * 100000


def load_dataset_cached(dataset_name: str) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    normalized_name = dataset_name.lower()
    if normalized_name not in DATASET_CACHE:
        x, y, feature_names = load_dataset(normalized_name)
        if issparse(x):
            x = x.tocsr()
        DATASET_CACHE[normalized_name] = (x, y, feature_names)
    return DATASET_CACHE[normalized_name]


def matrix_nonzero_count(x: np.ndarray) -> int:
    if issparse(x):
        return int(x.nnz)
    return int(np.count_nonzero(x))


def matrix_density(x: np.ndarray) -> float:
    total_cells = x.shape[0] * x.shape[1]
    if total_cells == 0:
        return 0.0
    return matrix_nonzero_count(x) / total_cells


def class_distribution_text(y: np.ndarray) -> str:
    labels, counts = np.unique(y, return_counts=True)
    distribution = {str(label): int(count) for label, count in zip(labels, counts)}
    return json.dumps(distribution, sort_keys=True)


def build_dataset_summary(dataset_names: list[str] = DATASET_NAMES) -> pd.DataFrame:
    rows: list[dict[str, object]] = []
    for dataset_name in dataset_names:
        x, y, _ = load_dataset_cached(dataset_name)
        rows.append(
            {
                "dataset": dataset_name,
                "rows": int(x.shape[0]),
                "features": int(x.shape[1]),
                "input_size": int(x.shape[0] * x.shape[1]),
                "format": "sparse" if issparse(x) else "dense",
                "nonzero_count": matrix_nonzero_count(x),
                "density": matrix_density(x),
                "class_distribution": class_distribution_text(y),
            }
        )
    return pd.DataFrame(rows)


def make_nested_stratified_subsets(
    y: np.ndarray,
    fractions: list[float] = FRACTIONS,
    random_state: int = RANDOM_STATE,
) -> dict[float, np.ndarray]:
    rng = np.random.default_rng(random_state)
    shuffled_by_class: dict[object, np.ndarray] = {}
    for label in np.unique(y):
        class_indices = np.flatnonzero(y == label)
        shuffled_by_class[label] = rng.permutation(class_indices)

    subsets: dict[float, np.ndarray] = {}
    for fraction in fractions:
        selected_parts: list[np.ndarray] = []
        for class_indices in shuffled_by_class.values():
            selected_count = int(round(len(class_indices) * fraction))
            selected_count = max(1, min(selected_count, len(class_indices)))
            selected_parts.append(class_indices[:selected_count])
        selected_indices = np.concatenate(selected_parts)
        selected_indices.sort()
        subsets[fraction] = selected_indices
    return subsets


def build_subsample_summary(dataset_name: str) -> pd.DataFrame:
    x, y, _ = load_dataset_cached(dataset_name)
    subsets = make_nested_stratified_subsets(y, random_state=dataset_seed(dataset_name))
    rows: list[dict[str, object]] = []
    for fraction, indices in subsets.items():
        subset_x = x[indices]
        subset_y = y[indices]
        rows.append(
            {
                "dataset": dataset_name,
                "fraction": fraction,
                "rows": int(subset_x.shape[0]),
                "features": int(subset_x.shape[1]),
                "input_size": int(subset_x.shape[0] * subset_x.shape[1]),
                "nonzero_count": matrix_nonzero_count(subset_x),
                "density": matrix_density(subset_x),
                "class_distribution": class_distribution_text(subset_y),
            }
        )
    return pd.DataFrame(rows)


def prepare_subset_dataset(dataset_name: str, indices: np.ndarray, random_state: int) -> PreparedDataset:
    x, y, feature_names = load_dataset_cached(dataset_name)
    subset_x = x[indices]
    subset_y = y[indices]

    x_train_validation, x_test, y_train_validation, y_test = train_test_split(
        subset_x,
        subset_y,
        test_size=0.20,
        random_state=random_state,
        stratify=subset_y,
    )
    x_train, x_validation, y_train, y_validation = train_test_split(
        x_train_validation,
        y_train_validation,
        test_size=0.25,
        random_state=random_state + 1,
        stratify=y_train_validation,
    )

    scaler = StandardScaler(with_mean=not issparse(x_train))
    x_train = scaler.fit_transform(x_train)
    x_validation = scaler.transform(x_validation)
    x_test = scaler.transform(x_test)

    return PreparedDataset(
        dataset_name=dataset_name,
        input_features=int(subset_x.shape[1]),
        x_train=x_train,
        x_validation=x_validation,
        x_test=x_test,
        y_train=y_train,
        y_validation=y_validation,
        y_test=y_test,
        feature_names=feature_names,
    )


## 3. Dataset Shape Summary

Run this before the algorithm experiments. The `input_size` column is `rows * features`, while `nonzero_count` and `density` help explain sparse datasets where rectangular input size alone may not predict runtime.

In [ ]:
dataset_summary = build_dataset_summary()
dataset_summary.to_csv(RESULTS_DIR / "dataset_summary.csv", index=False)
display(dataset_summary)


## 4. Full-Dataset Linear SVM Training Time

This measures one full-feature evaluator-model training run per dataset. The metaheuristic runtime below is still the main execution-time result, but this table helps explain whether repeated SVM training inside the fitness function is likely to dominate runtime.

In [ ]:
def time_full_dataset_svm_training(dataset_names: list[str] = DATASET_NAMES) -> pd.DataFrame:
    rows: list[dict[str, object]] = []
    for dataset_name in dataset_names:
        x, y, _ = load_dataset_cached(dataset_name)
        seed = dataset_seed(dataset_name)
        x_train, _, y_train, _ = train_test_split(
            x,
            y,
            test_size=0.40,
            random_state=seed,
            stratify=y,
        )
        scaler = StandardScaler(with_mean=not issparse(x_train))
        x_train = scaler.fit_transform(x_train)
        model = SGDClassifier(
            loss="hinge",
            alpha=0.0001,
            max_iter=1000,
            tol=1e-3,
            random_state=42,
        )
        start = time.perf_counter()
        model.fit(x_train, y_train)
        training_time_seconds = time.perf_counter() - start
        rows.append(
            {
                "dataset": dataset_name,
                "train_rows": int(x_train.shape[0]),
                "features": int(x_train.shape[1]),
                "train_input_size": int(x_train.shape[0] * x_train.shape[1]),
                "training_time_seconds": training_time_seconds,
            }
        )
    return pd.DataFrame(rows)


svm_training_times = time_full_dataset_svm_training()
svm_training_times.to_csv(RESULTS_DIR / "full_dataset_svm_training_times.csv", index=False)
display(svm_training_times)


## 5. Runtime Experiment Helpers

In [ ]:
def run_single_algorithm_for_subset(
    dataset_name: str,
    fraction: float,
    subset_indices: np.ndarray,
    algorithm_name: str,
    algorithm_fn: Callable[..., tuple[np.ndarray, float, list[float], bool]],
    fraction_index: int,
    algorithm_index: int,
) -> dict[str, object]:
    seed = dataset_seed(dataset_name) + fraction_index * 1000 + algorithm_index
    prepared_dataset = prepare_subset_dataset(dataset_name, subset_indices, seed)
    rng = np.random.default_rng(seed)

    fitness_fn, validation_accuracy_fn = make_fitness_function(
        prepared_dataset.x_train,
        prepared_dataset.x_validation,
        prepared_dataset.y_train,
        prepared_dataset.y_validation,
        rng,
    )

    print(f"Running {algorithm_name} on {dataset_name}, fraction={fraction:.1f}, rows={len(subset_indices)}")
    start = time.perf_counter()
    best_mask, best_fitness, convergence_history, stopped_early = algorithm_fn(
        prepared_dataset.input_features,
        fitness_fn,
        rng,
        POPULATION_SIZE,
        ITERATIONS,
        progress_callback=lambda iteration, score: print(
            f"  iteration {iteration}/{ITERATIONS}, best_fitness={score:.6f}",
            flush=True,
        ),
        early_stopping_patience=EARLY_STOPPING_PATIENCE,
        min_delta=MIN_DELTA,
    )
    runtime_seconds = time.perf_counter() - start

    validation_accuracy = validation_accuracy_fn(best_mask)
    test_accuracy_fn = make_accuracy_function(
        prepared_dataset.x_train,
        prepared_dataset.y_train,
        prepared_dataset.x_test,
        prepared_dataset.y_test,
        rng,
    )
    test_accuracy = test_accuracy_fn(best_mask)

    x, _, _ = load_dataset_cached(dataset_name)
    subset_x = x[subset_indices]

    return {
        "dataset": dataset_name,
        "algorithm": algorithm_name,
        "fraction": fraction,
        "rows": int(subset_x.shape[0]),
        "features": int(subset_x.shape[1]),
        "input_size": int(subset_x.shape[0] * subset_x.shape[1]),
        "nonzero_count": matrix_nonzero_count(subset_x),
        "density": matrix_density(subset_x),
        "runtime_seconds": runtime_seconds,
        "best_fitness": best_fitness,
        "validation_accuracy": validation_accuracy,
        "test_accuracy": test_accuracy,
        "selected_count": int(best_mask.sum()),
        "iterations_run": max(len(convergence_history) - 1, 0),
        "stopped_early": stopped_early,
    }


def run_dataset_scalability(dataset_name: str) -> pd.DataFrame:
    x, y, _ = load_dataset_cached(dataset_name)
    subsets = make_nested_stratified_subsets(y, random_state=dataset_seed(dataset_name))

    print(f"Class-ratio check for {dataset_name}")
    subsample_summary = build_subsample_summary(dataset_name)
    display(subsample_summary)
    subsample_summary.to_csv(RESULTS_DIR / f"{dataset_name}_subsample_summary.csv", index=False)

    result_rows: list[dict[str, object]] = []
    for fraction_index, (fraction, indices) in enumerate(subsets.items()):
        for algorithm_index, (algorithm_name, algorithm_fn) in enumerate(ALGORITHMS.items()):
            result = run_single_algorithm_for_subset(
                dataset_name,
                fraction,
                indices,
                algorithm_name,
                algorithm_fn,
                fraction_index,
                algorithm_index,
            )
            result_rows.append(result)
            pd.DataFrame(result_rows).to_csv(RESULTS_DIR / f"{dataset_name}_scalability_results.csv", index=False)
            print(
                f"  done: validation_accuracy={result['validation_accuracy']:.4f}, "
                f"test_accuracy={result['test_accuracy']:.4f}, "
                f"selected={result['selected_count']}, "
                f"runtime={result['runtime_seconds']:.4f}s",
                flush=True,
            )

    dataset_results = pd.DataFrame(result_rows)
    display(dataset_results)
    return dataset_results


## 6. Dataset Runs

Each dataset has its own cell. If a large dataset takes too long, completed dataset CSV files remain available under `results/time_complexity_analysis/`.

### Madelon

In [ ]:
madelon_results = run_dataset_scalability("madelon")


### Arcene

In [ ]:
arcene_results = run_dataset_scalability("arcene")


### Dexter

In [ ]:
dexter_results = run_dataset_scalability("dexter")


### Dorothea

In [ ]:
dorothea_results = run_dataset_scalability("dorothea")


### Gisette

In [ ]:
gisette_results = run_dataset_scalability("gisette")


## 7. Combine Saved Results and Plot Execution Time

In [ ]:
def load_saved_scalability_results() -> pd.DataFrame:
    result_files = sorted(RESULTS_DIR.glob("*_scalability_results.csv"))
    if not result_files:
        raise FileNotFoundError(f"No scalability result files found in {RESULTS_DIR}")
    combined = pd.concat((pd.read_csv(path) for path in result_files), ignore_index=True)
    combined.to_csv(RESULTS_DIR / "combined_scalability_results.csv", index=False)
    return combined


def plot_execution_time_by_dataset(results: pd.DataFrame) -> None:
    datasets = [name for name in DATASET_NAMES if name in set(results["dataset"])]
    if not datasets:
        raise ValueError("No dataset results available to plot.")

    column_count = 2
    row_count = int(np.ceil(len(datasets) / column_count))
    figure, axes = plt.subplots(row_count, column_count, figsize=(14, 5 * row_count), squeeze=False)
    axes_flat = axes.ravel()

    for axis_index, dataset_name in enumerate(datasets):
        axis = axes_flat[axis_index]
        dataset_results = results[results["dataset"] == dataset_name]
        for algorithm_name in ALGORITHMS:
            algorithm_results = dataset_results[dataset_results["algorithm"] == algorithm_name].sort_values("input_size")
            if algorithm_results.empty:
                continue
            axis.plot(
                algorithm_results["input_size"],
                algorithm_results["runtime_seconds"],
                marker="o",
                label=algorithm_name,
            )
        axis.set_title(dataset_name)
        axis.set_xlabel("Input size (rows x features)")
        axis.set_ylabel("Execution time (seconds)")
        axis.grid(True, alpha=0.3)
        axis.legend()

    for empty_axis in axes_flat[len(datasets):]:
        empty_axis.axis("off")

    figure.tight_layout()
    output_path = RESULTS_DIR / "execution_time_vs_input_size_by_dataset.png"
    figure.savefig(output_path, dpi=200)
    plt.show()
    print(f"Saved plot: {output_path}")


def plot_combined_execution_time(results: pd.DataFrame) -> None:
    figure, axis = plt.subplots(figsize=(12, 7))
    for algorithm_name in ALGORITHMS:
        algorithm_results = results[results["algorithm"] == algorithm_name].sort_values("input_size")
        if algorithm_results.empty:
            continue
        axis.scatter(
            algorithm_results["input_size"],
            algorithm_results["runtime_seconds"],
            label=algorithm_name,
            alpha=0.8,
        )
    axis.set_xscale("log")
    axis.set_xlabel("Input size (rows x features, log scale)")
    axis.set_ylabel("Execution time (seconds)")
    axis.set_title("Execution Time vs Input Size")
    axis.grid(True, alpha=0.3)
    axis.legend()
    figure.tight_layout()
    output_path = RESULTS_DIR / "execution_time_vs_input_size_combined.png"
    figure.savefig(output_path, dpi=200)
    plt.show()
    print(f"Saved plot: {output_path}")


combined_results = load_saved_scalability_results()
display(combined_results)
plot_execution_time_by_dataset(combined_results)
plot_combined_execution_time(combined_results)
